In [65]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from catboost import CatBoostRegressor, Pool
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

In [2]:
df1 = pd.read_excel(r"../india_weather_rainfall_data.xlsx")
df1.head()

,date_of_record,month,season,station_name,state,district,avg_temp,min_temp,max_temp,wind_speed,air_pressure,elevation,latitude,longitude,rainfall
0,2021-01-02,January,Winter,Gulmarg,JK,Baramulla,-2.2,-6.6,-0.8,2.2,1020.0,2652,34.05,74.4,0.1
1,2021-01-03,January,Winter,Gulmarg,JK,Baramulla,-3.6,-4.6,-1.8,3.7,1019.5,2652,34.05,74.4,4.4
2,2021-01-04,January,Winter,Gulmarg,JK,Baramulla,-3.0,-4.5,-1.1,2.1,1022.0,2652,34.05,74.4,2.3
3,2021-01-05,January,Winter,Gulmarg,JK,Baramulla,-3.3,-5.1,-1.2,2.8,1015.6,2652,34.05,74.4,35.0
4,2021-01-06,January,Winter,Gulmarg,JK,Baramulla,-3.9,-8.3,-1.0,3.4,1015.3,2652,34.05,74.4,25.5


In [161]:
df2 = df1.copy()

In [162]:
df2["date_of_record"] = pd.to_datetime(df2["date_of_record"])
df2["day_of_year"] = df2["date_of_record"].dt.dayofyear
df2["year"] = df2["date_of_record"].dt.year
df2.head(5)

,date_of_record,month,season,station_name,state,district,avg_temp,min_temp,max_temp,wind_speed,air_pressure,elevation,latitude,longitude,rainfall,day_of_year,year
0,2021-01-02,January,Winter,Gulmarg,JK,Baramulla,-2.2,-6.6,-0.8,2.2,1020.0,2652,34.05,74.4,0.1,2,2021
1,2021-01-03,January,Winter,Gulmarg,JK,Baramulla,-3.6,-4.6,-1.8,3.7,1019.5,2652,34.05,74.4,4.4,3,2021
2,2021-01-04,January,Winter,Gulmarg,JK,Baramulla,-3.0,-4.5,-1.1,2.1,1022.0,2652,34.05,74.4,2.3,4,2021
3,2021-01-05,January,Winter,Gulmarg,JK,Baramulla,-3.3,-5.1,-1.2,2.8,1015.6,2652,34.05,74.4,35.0,5,2021
4,2021-01-06,January,Winter,Gulmarg,JK,Baramulla,-3.9,-8.3,-1.0,3.4,1015.3,2652,34.05,74.4,25.5,6,2021


In [163]:
df2.isna().sum()

date_of_record         0
month                  0
season                 0
station_name           0
state                  0
district               0
avg_temp               0
min_temp           43898
max_temp          110598
wind_speed        274444
air_pressure      304664
elevation              0
latitude               0
longitude              0
rainfall          257554
day_of_year            0
year                   0
dtype: int64

In [164]:
# Keep min_temp and max_temp for target prediction, but do not use them as input features.
# This also fixes notebook state if an earlier run already dropped these columns from df2.
for temp_col in ["min_temp", "max_temp"]:
    if temp_col not in df2.columns:
        df2[temp_col] = df1[temp_col]

df2.shape

(970339, 17)

In [165]:
for i, j in zip(df2.columns, df2.isnull().sum()):
    if j:
        print(f"{i}: {round(j/df2.shape[0]*100, 3)}%")

min_temp: 4.524%
max_temp: 11.398%
wind_speed: 28.283%
air_pressure: 31.398%
rainfall: 26.543%


In [166]:
STATION_GROUP = ["state", "district", "station_name"]

df2 = df2.sort_values([*STATION_GROUP, "date_of_record"])

df2["sin_day"] = np.sin(2 * np.pi * df2["day_of_year"] / 365.25)
df2["cos_day"] = np.cos(2 * np.pi * df2["day_of_year"] / 365.25)

for lag in [1, 3, 7]:
    df2[f"temp_lag_{lag}"] = df2.groupby(STATION_GROUP)["avg_temp"].shift(lag)
    df2[f"temp_max_lag_{lag}"] = df2.groupby(STATION_GROUP)["max_temp"].shift(lag)
    df2[f"rain_lag_{lag}"] = df2.groupby(STATION_GROUP)["rainfall"].shift(lag)

# Keep rows with missing weather values; the model pipeline will impute them.
df2.isna().sum()

date_of_record         0
month                  0
season                 0
station_name           0
state                  0
district               0
avg_temp               0
min_temp           43898
max_temp          110598
wind_speed        274444
air_pressure      304664
elevation              0
latitude               0
longitude              0
rainfall          257554
day_of_year            0
year                   0
sin_day                0
cos_day                0
temp_lag_1             1
temp_lag_3             3
temp_lag_7             7
temp_max_lag_1    110599
temp_max_lag_3    110601
temp_max_lag_7    110605
rain_lag_1        257555
rain_lag_3        257557
rain_lag_7        257561
dtype: int64

In [167]:
df3 = df2.copy()

In [168]:
df3 = df3.sort_values("date_of_record")
cutoff_date = "2024-01-01"

train = df3[df3["date_of_record"] < cutoff_date]
test = df3[df3["date_of_record"] >= cutoff_date]

train.shape, test.shape

((805740, 28), (164599, 28))

In [169]:
feature_cols = [
    "year", "sin_day", "cos_day", "rainfall", "wind_speed", "air_pressure",
    "elevation", "latitude", "longitude", "month", "season", "state",
    "district", "station_name", "temp_lag_1", "temp_lag_3", "temp_lag_7",
    "temp_max_lag_1", "temp_max_lag_3", "temp_max_lag_7",
    "rain_lag_1", "rain_lag_3", "rain_lag_7"
]
target_cols = ["avg_temp", "min_temp", "max_temp"]

X_train = train[feature_cols]
X_test = test[feature_cols]

X_train.shape, X_test.shape

((805740, 23), (164599, 23))

In [170]:
numeric_features = [
    "year", "sin_day", "cos_day", "rainfall", "wind_speed", "air_pressure",
    "elevation", "latitude", "longitude", "temp_lag_1", "temp_lag_3", "temp_lag_7",
    "temp_max_lag_1", "temp_max_lag_3", "temp_max_lag_7",
    "rain_lag_1", "rain_lag_3", "rain_lag_7"
]
categorical_features = ["month", "season", "state", "district", "station_name"]

# Restore target columns if train/test were created before min_temp and max_temp were kept.
train = train.copy()
test = test.copy()
for temp_col in ["min_temp", "max_temp"]:
    if temp_col not in train.columns:
        train[temp_col] = df1.loc[train.index, temp_col]
    if temp_col not in test.columns:
        test[temp_col] = df1.loc[test.index, temp_col]


def build_model():
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), numeric_features),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
                    ]
                ),
                categorical_features,
            ),
        ]
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "regressor",
                HistGradientBoostingRegressor(
                    max_iter=220,
                    learning_rate=0.08,
                    l2_regularization=0.01,
                    random_state=42,
                ),
            ),
        ]
    )


models = {}
metrics = []

for target_col in target_cols:
    train_mask = train[target_col].notna()
    test_mask = test[target_col].notna()

    target_model = build_model()
    target_model.fit(train.loc[train_mask, feature_cols], train.loc[train_mask, target_col])

    predictions = target_model.predict(test.loc[test_mask, feature_cols])
    y_test = test.loc[test_mask, target_col]

    models[target_col] = target_model
    metrics.append(
        {
            "target": target_col,
            "train_rows": train_mask.sum(),
            "test_rows": test_mask.sum(),
            "MAE": mean_absolute_error(y_test, predictions),
            "RMSE": mean_squared_error(y_test, predictions) ** 0.5,
            "R2": r2_score(y_test, predictions),
        }
    )

metrics_df = pd.DataFrame(metrics)
print(metrics_df)

     target  train_rows  test_rows       MAE      RMSE        R2
0  avg_temp      805740     164599  0.661878  0.933311  0.974359
1  min_temp      761924     164517  0.957077  1.296585  0.958467
2  max_temp      695495     164246  1.004602  1.384136  0.942026


In [172]:
current_date = pd.Timestamp("2026-07-02")
current_day_of_year = current_date.dayofyear

kolkata_today = pd.DataFrame(
    [
        {
            "year": current_date.year,
            "sin_day": np.sin(2 * np.pi * current_day_of_year / 365.25),
            "cos_day": np.cos(2 * np.pi * current_day_of_year / 365.25),

            # Weather features
            "rainfall": 0.0,         # Today's estimated rainfall (mm)
            "wind_speed": 5.0,       # km/h
            "air_pressure": 1001.0,   # hPa
            "elevation": 5,
            "latitude": 22.5333,
            "longitude": 88.3333,

            # Categorical features
            "month": "July",
            "season": "Monsoon",
            "state": "WB",
            "district": "Kolkata",
            "station_name": "Calcutta / Alipore",

            # Average temperature lags
            "temp_lag_1": 30,
            "temp_lag_3": 28,
            "temp_lag_7": 31,

            # Maximum temperature lags
            "temp_max_lag_1": 35,
            "temp_max_lag_3": 33,
            "temp_max_lag_7": 35,

            # Rainfall lags (mm)
            "rain_lag_1": 3.8,
            "rain_lag_3": 27.9,
            "rain_lag_7": 84.6,
        }
    ]
)

kolkata_predictions = {
    target_col: models[target_col].predict(kolkata_today[feature_cols])[0]
    for target_col in target_cols
}

print("Predicted temperatures for Kolkata today:")
print(f"Average: {kolkata_predictions['avg_temp']:.2f}°C")
print(f"Minimum: {kolkata_predictions['min_temp']:.2f}°C")
print(f"Maximum: {kolkata_predictions['max_temp']:.2f}°C")

pd.DataFrame([kolkata_predictions])

Predicted temperatures for Kolkata today:
Average: 30.67°C
Minimum: 27.31°C
Maximum: 35.25°C


,avg_temp,min_temp,max_temp
0,30.665415,27.314897,35.247339


In [159]:
def build_model_2():
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), numeric_features),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
                    ]
                ),
                categorical_features,
            ),
        ]
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "regressor",
                CatBoostRegressor(
                    iterations=1000,
                    learning_rate=0.1,
                    depth=6,
                    loss_function='RMSE',
                    eval_metric='RMSE',
                    random_seed=42,
                    verbose=100
                ),
            ),
        ]
    )

models_2 = {}
metrics_2 = []

for target_col in target_cols:
    train_mask = train[target_col].notna()
    test_mask = test[target_col].notna()

    target_model = build_model_2()
    target_model.fit(train.loc[train_mask, feature_cols], train.loc[train_mask, target_col])

    predictions = target_model.predict(test.loc[test_mask, feature_cols])
    y_test = test.loc[test_mask, target_col]

    models_2[target_col] = target_model
    metrics_2.append(
        {
            "target": target_col,
            "train_rows": train_mask.sum(),
            "test_rows": test_mask.sum(),
            "MAE": mean_absolute_error(y_test, predictions),
            "RMSE": mean_squared_error(y_test, predictions) ** 0.5,
            "R2": r2_score(y_test, predictions),
        }
    )

metrics_2_df = pd.DataFrame(metrics_2)
print(metrics_2_df)

0:	learn: 4.8678643	total: 31ms	remaining: 31s
100:	learn: 1.0641678	total: 1.71s	remaining: 15.2s
200:	learn: 1.0395124	total: 3.5s	remaining: 13.9s
300:	learn: 1.0241843	total: 5.14s	remaining: 11.9s
400:	learn: 1.0135382	total: 6.77s	remaining: 10.1s
500:	learn: 1.0048524	total: 8.4s	remaining: 8.37s
600:	learn: 0.9979354	total: 10s	remaining: 6.67s
700:	learn: 0.9917222	total: 11.7s	remaining: 4.98s
800:	learn: 0.9860125	total: 13.3s	remaining: 3.31s
900:	learn: 0.9808763	total: 14.9s	remaining: 1.64s
999:	learn: 0.9760791	total: 16.6s	remaining: 0us
0:	learn: 5.4288249	total: 21.7ms	remaining: 21.6s
100:	learn: 1.3691867	total: 1.61s	remaining: 14.4s
200:	learn: 1.2839196	total: 3.2s	remaining: 12.7s
300:	learn: 1.2429221	total: 4.81s	remaining: 11.2s
400:	learn: 1.2142846	total: 6.45s	remaining: 9.64s
500:	learn: 1.1938246	total: 8.1s	remaining: 8.07s
600:	learn: 1.1766845	total: 9.72s	remaining: 6.45s
700:	learn: 1.1637516	total: 11.3s	remaining: 4.83s
800:	learn: 1.1522910	tota

In [171]:
current_date = pd.Timestamp("2026-07-02")
current_day_of_year = current_date.dayofyear

kolkata_today = pd.DataFrame(
    [
        {
            "year": current_date.year,
            "sin_day": np.sin(2 * np.pi * current_day_of_year / 365.25),
            "cos_day": np.cos(2 * np.pi * current_day_of_year / 365.25),

            # Weather features
            "rainfall": 5.2,         # Today's estimated rainfall (mm)
            "wind_speed": 8.0,       # km/h
            "air_pressure": 999.0,   # hPa
            "elevation": 5,
            "latitude": 22.5333,
            "longitude": 88.3333,

            # Categorical features
            "month": "July",
            "season": "Monsoon",
            "state": "WB",
            "district": "Kolkata",
            "station_name": "Calcutta / Alipore",

            # Average temperature lags
            "temp_lag_1": 31.4,
            "temp_lag_3": 30.8,
            "temp_lag_7": 29.9,

            # Maximum temperature lags
            "temp_max_lag_1": 35.1,
            "temp_max_lag_3": 34.5,
            "temp_max_lag_7": 33.8,

            # Rainfall lags (mm)
            "rain_lag_1": 12.4,
            "rain_lag_3": 4.8,
            "rain_lag_7": 18.6,
        }
    ]
)

kolkata_predictions = {
    target_col: models_2[target_col].predict(kolkata_today[feature_cols])[0]
    for target_col in target_cols
}

print("Predicted temperatures for Kolkata today:")
print(f"Average: {kolkata_predictions['avg_temp']:.2f}°C")
print(f"Minimum: {kolkata_predictions['min_temp']:.2f}°C")
print(f"Maximum: {kolkata_predictions['max_temp']:.2f}°C")

pd.DataFrame([kolkata_predictions])

Predicted temperatures for Kolkata today:
Average: 30.65°C
Minimum: 27.60°C
Maximum: 34.74°C


,avg_temp,min_temp,max_temp
0,30.650092,27.604419,34.743263
